# Gold: lab_test_trends

Builds `ws_healthcare_cntrl_india.gold.lab_test_trends` (one row per test type per calendar month) by aggregating silver lab_results, enriched with calendar month/quarter names, with a LAG-based month-over-month abnormal-rate trend. Per STTM `sttm_silver_to_gold.xlsx` (sheet `lab_test_trends`); full rebuild.

In [0]:
%sql
CREATE OR REPLACE TABLE ws_healthcare_cntrl_india.gold.lab_test_trends
COMMENT 'Gold lab test trends: one row per test_name per calendar month with abnormal/critical rates and month-over-month trend. Source: STTM silver_to_gold, sheet lab_test_trends.'
AS
WITH monthly AS (
  SELECT
    test_name,
    year(test_date)                           AS year,
    month(test_date)                          AS month,
    COUNT(*)                                  AS total_tests,
    COUNT_IF(result_status = 'Abnormal')      AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')      AS critical_test_count
  FROM ws_healthcare_cntrl_india.silver.lab_results
  GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
  SELECT
    m.*,
    CAST(m.abnormal_test_count * 100.0 / m.total_tests AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(m.critical_test_count * 100.0 / m.total_tests AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
with_lag AS (
  SELECT
    r.*,
    LAG(r.abnormal_rate_pct) OVER (PARTITION BY r.test_name ORDER BY r.year, r.month) AS prev_month_abnormal_rate_pct
  FROM rates r
)
SELECT
  w.test_name,
  CAST(w.year AS INT)                                                       AS year,
  CAST(w.month AS INT)                                                      AS month,
  c.month_name,
  c.quarter_name,
  CAST(w.total_tests AS INT)                                                AS total_tests,
  CAST(w.abnormal_test_count AS INT)                                        AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                                        AS critical_test_count,
  w.abnormal_rate_pct,
  w.critical_rate_pct,
  CAST(w.prev_month_abnormal_rate_pct AS DECIMAL(5,2))                      AS prev_month_abnormal_rate_pct,
  CAST(w.abnormal_rate_pct - w.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct  -- NULL when no prior month
FROM with_lag w
LEFT JOIN ws_healthcare_cntrl_india.silver.calendar c
  ON c.date_key = CAST(date_format(make_date(w.year, w.month, 1), 'yyyyMMdd') AS INT);

In [0]:
# Return the gold table record count to the caller (e.g. a Job or dbutils.notebook.run)
updated_df = spark.table("ws_healthcare_cntrl_india.gold.lab_test_trends")
record_count = updated_df.count()
print(f"gold.lab_test_trends record count: {record_count}")

dbutils.notebook.exit(str(record_count))